# Three controlled feature experiments — Leonard

Compare remaining team strength, advantage trajectory, and boost threat independently. LR starts from Aimilia’s original 106 features; HGB starts from the previous 217-feature model. Model settings stay fixed.

**Validation:** five stratified group folds on 7,996 development battles. Exact duplicate observations stay together. Exclude all 2,000 battles from the previously opened holdout plus four development duplicates of them. No test or holdout evaluation. These are exploratory CV results on previously explored development data.

Default: review the completed run. Set `RUN_NEW_EXPERIMENT=True` to execute all eight candidates (40 fits) again. The three additions are tested independently, without combinations or parameter search.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pokemon_challenge").is_dir())
sys.path.insert(0, str(ROOT / "notebooks/leonard"))
from run_progressive_experiments import run
RUN_NEW_EXPERIMENT = False
RUN_FOLDER = ROOT / "outputs/leonard/20261102_121822_824155_progressive_features"
if RUN_NEW_EXPERIMENT:
    RUN_FOLDER = run()
print("Results:", RUN_FOLDER)


## Paired comparisons

Each addition is compared against the baseline for the same model on the same folds. Deltas are percentage points. New folds differ from notebook 05, so compare against these newly measured baselines.


In [ ]:
results = pd.read_csv(RUN_FOLDER / "cv_results.csv")
paired = pd.read_csv(RUN_FOLDER / "paired_comparison.csv")
print(results[["candidate", "input_features", "cv_accuracy", "fold_std"]].to_string(index=False))
print(paired[["candidate", "delta_pp", "folds_better", "folds_worse", "corrected_errors", "introduced_errors"]].to_string(index=False))


## Interpretation of the completed run

* Remaining strength: LR 65.995% (−0.325 pp); HGB 66.846% (+0.025 pp). No useful gain demonstrated. Opponent stats remain known only for its lead; no inferred species statistics are fitted.
* Advantage trajectory: LR 66.396% (+0.075 pp); HGB 66.646% (−0.175 pp). The LR change is very small. Existing 10/20/30 roster values are excluded from the added group to avoid duplicating them.
* Boost threat: LR 66.283% (−0.037 pp); HGB **67.234% (+0.413 pp)**, better in 4/5 folds. Corrects 292 baseline mistakes and introduces 259, for 33 net additional correct predictions among 7,996 development battles.

**Recommendation:** retain boost threat as the next HGB feature candidate; defer the other two groups. This is promising CV evidence, not an independent performance confirmation. Last-seen HP, status and boosts can be stale; boosts can reset on switching. Related nonidentical battles are not covered by the exact-duplicate grouping. No new Kaggle CSV is generated by this validation experiment.


In [ ]:
manifest = json.loads((RUN_FOLDER / "experiment.json").read_text())
print(json.dumps({k: manifest[k] for k in ("cv", "folds", "development_rows", "previous_holdout_rows", "quarantined_duplicate_rows", "splits_sha256", "holdout_used", "test_used")}, indent=2))
